<a href="https://colab.research.google.com/github/samdvey/MKTG6203DigitalTwins/blob/main/(8)_InfluencerWealth_Phase2_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2: RAG-Grounded Synthetic Interviews — Influencer Wealth Perception
### Using Real Twin-2K-500 Participants as Digital Twins
**Professor Joseph Pancras | University of Connecticut School of Business**

---

## What is RAG, and why does it matter here?

**RAG = Retrieval-Augmented Generation.** Instead of asking GPT-4 to generate a response from scratch, we first *retrieve* a real participant's full psychological profile and *augment* the prompt with it before generating the response.

In Phase 1, you built five fictional personas and assigned their traits by hand — Zoe had high self-monitoring, Marcus had low self-monitoring, and so on. This was fast and educational, but it had important limitations:

1. **The traits were researcher-invented, not measured.** There is no survey behind Zoe's "high self-monitoring" label.
2. **The profile was thin.** A few sentences cannot capture whether someone is also high in need for cognition, financially literate, or politically conservative — all of which may shape how they process influencer wealth signals.
3. **The responses could not be verified.** If Zoe said she loves wealth displays, you could not check whether a real high-self-monitoring person would actually respond that way.

**Phase 2 solves all three problems.** Each Twin-2K-500 `persona_summary` is derived from 500+ validated survey questions. When this feeds GPT-4 as the system prompt, the AI response is grounded in real data — self-monitoring measured by Snyder's scale, need for uniqueness by a validated instrument, minimalism by a dedicated battery.

---

## What changed from Phase 1?

| | Phase 1 | Phase 2 (this notebook) |
|---|---|---|
| **Personas** | Hand-crafted fictional characters | Real Twin-2K-500 participants |
| **Profile depth** | 3-5 sentences | Full 500+ question survey profile |
| **Trait grounding** | Researcher-assigned labels | Validated psychometric scores |
| **Verifiability** | Cannot check | Can look up participant by PID |
| **Selection method** | Researcher judgment | Nearest-neighbor matching on trait scores |

---

## Relevant Twin-2K-500 traits for influencer wealth research

Three traits drive participant selection in this notebook, matching the Phase 1 persona design:

- **Self-monitoring** (`score_selfmonitoring`): Sensitivity to social cues and status signals. High self-monitors notice and process influencer wealth displays more intensely.
- **Need for uniqueness** (`score_needforuniqueness`): Desire to differentiate from the mainstream. High scorers are more likely to reject influencers they perceive as commercially co-opted.
- **Minimalism** (`score_minimalism`): Preference for simplicity and skepticism toward conspicuous consumption. High scorers are more critical of wealth displays.

The theoretically interesting profile is **high self-monitoring + high need for uniqueness + high minimalism**: someone who notices wealth signals intensely but is motivated to reject them. This is the Aaliyah archetype from Phase 1 — but now grounded in a real person's measured scores.

---

## Pipeline (7 cells)

| Cell | What it does |
|------|--------------|
| Cell 1 | Install packages |
| Cell 2 | Enter API key |
| Cell 3 | Load 2,058 participants from Hugging Face |
| Cell 4 | Extract psychological scores |
| Cell 5 | Select 5 participants spanning the trait space |
| Cell 6 | Run RAG-grounded interviews (same 3 questions as Phase 1) |
| Cell 7 | Display results with Phase 1 comparison guidance |

In [ ]:
# CELL 1: Install required packages
# This may take 1-2 minutes. Lots of text scrolling is normal.

!pip install openai datasets pandas --quiet
print("")

In [ ]:
# CELL 2: Enter your OpenAI API key
# When you run this cell, a box will appear. Paste your key there and press Enter.
# Your key will NOT be visible on screen.

import getpass
import openai

api_key = getpass.getpass("")
client = openai.OpenAI(api_key=api_key)
print("")

In [ ]:
# CELL 3: Load all 2,058 participants from Twin-2K-500
#
# WHY RAG NEEDS THE FULL DATASET:
# Phase 1 defined 5 personas by hand. Here we load all 2,058 real participants
# so we can SEARCH for the ones whose actual measured traits best match the
# psychological profiles most relevant to influencer wealth perception research.
# This is the "retrieval" step in RAG.
#
# Runtime: ~1-2 minutes. Watch for the progress bars.

import pandas as pd
from datasets import load_dataset

print("Loading Twin-2K-500 dataset from Hugging Face...")
print("(Download progress bars will appear — this is normal)")
print()

chunk_files = [
    "full_persona/chunks/persona_chunk_001.parquet",
    "full_persona/chunks/persona_chunk_002.parquet",
    "full_persona/chunks/persona_chunk_003.parquet",
    "full_persona/chunks/persona_chunk_004.parquet",
    "full_persona/chunks/persona_chunk_005.parquet",
    "full_persona/chunks/persona_chunk_006.parquet",
    "full_persona/chunks/persona_chunk_007.parquet",
]

all_dfs = []
for i, chunk_file in enumerate(chunk_files, 1):
    ds = load_dataset(
        "LLM-Digital-Twin/Twin-2K-500",
        data_files=chunk_file,
        split="train"
    )
    df_chunk = ds.to_pandas()
    all_dfs.append(df_chunk)
    print(f"  Chunk {i}/7 loaded: {len(df_chunk)} participants")

df = pd.concat(all_dfs, ignore_index=True)
print()
print(f"SUCCESS: Loaded {len(df):,} participants total")
print(f"Columns: {list(df.columns)}")

In [ ]:
# ============================================================================
# REPLACE CELL 4 WITH THIS - Using Traits That Actually Exist
# ============================================================================

import re
import numpy as np

def extract_score(text, score_name):
    """Extracts numeric score from persona_summary text."""
    if pd.isna(text):
        return None
    pattern = rf"{score_name}\s*=\s*([\-0-9\.]+)"
    match = re.search(pattern, text, re.IGNORECASE)
    if match:
        try:
            return float(match.group(1))
        except:
            return None
    return None

def extract_percentile(text, score_name):
    """Extracts percentile rank from persona_summary text."""
    if pd.isna(text):
        return None
    pattern = rf"{score_name}\s*=\s*[\-0-9\.]+\s*\((\d+)(?:st|nd|rd|th) percentile\)"
    match = re.search(pattern, text, re.IGNORECASE)
    if match:
        try:
            return int(match.group(1))
        except:
            return None
    return None

print("Extracting psychological traits for influencer wealth research...")
print()
print("TRAIT SELECTION RATIONALE:")
print("-" * 70)
print("For studying influencer wealth perception, we use:")
print()
print("1. EXTRAVERSION (score_extraversion)")
print("   High = outgoing, socially engaged, status-aware")
print("   Low = reserved, less influenced by social trends")
print("   → Predicts engagement with influencer content")
print()
print("2. AGREEABLENESS (score_agreeableness)")
print("   High = trusting, conforming, follows social norms")
print("   Low = skeptical, critical, questions motives")
print("   → Predicts whether wealth displays are seen as aspirational or suspicious")
print()
print("3. MINIMALISM (score_minimalism)")
print("   High = anti-consumerism, values simplicity")
print("   Low = materialistic, values possessions/status")
print("   → Directly measures attitude toward conspicuous consumption")
print()
print("These three traits create meaningful segments:")
print("  - High E + Low A + Low M = Status aspirational (wants influencer lifestyle)")
print("  - Low E + High A + High M = Anti-materialist (rejects wealth displays)")
print("  - High E + Low A + High M = Conflicted critic (notices but disapproves)")
print("=" * 70)
print()

# Extract the three traits that EXIST in the dataset
df["extraversion"] = df["persona_summary"].apply(lambda x: extract_score(x, "score_extraversion"))
df["agreeableness"] = df["persona_summary"].apply(lambda x: extract_score(x, "score_agreeableness"))
df["minimalism"] = df["persona_summary"].apply(lambda x: extract_score(x, "score_minimalism"))

df["extra_pct"] = df["persona_summary"].apply(lambda x: extract_percentile(x, "score_extraversion"))
df["agree_pct"] = df["persona_summary"].apply(lambda x: extract_percentile(x, "score_agreeableness"))
df["min_pct"] = df["persona_summary"].apply(lambda x: extract_percentile(x, "score_minimalism"))

# Clean dataset
df_clean = df.dropna(subset=["extraversion", "agreeableness", "minimalism"]).copy()

print(f"Participants with complete scores: {len(df_clean):,} of {len(df):,}")
print()
print("Score ranges:")
print(f"  Extraversion:  min={df_clean['extraversion'].min():.2f}, max={df_clean['extraversion'].max():.2f}")
print(f"  Agreeableness: min={df_clean['agreeableness'].min():.2f}, max={df_clean['agreeableness'].max():.2f}")
print(f"  Minimalism:    min={df_clean['minimalism'].min():.2f}, max={df_clean['minimalism'].max():.2f}")
print()
print("✓ Ready to select participants!")

In [ ]:

# ============================================================================
# REPLACE CELL 5 WITH THIS - Participant Selection
# ============================================================================

print()
print("=" * 70)
print("SELECTING 5 PARTICIPANTS - PHASE 2 DIGITAL TWIN ARCHETYPES")
print("=" * 70)
print()

# Normalize scores to 0-1 range
for col in ["extraversion", "agreeableness", "minimalism"]:
    col_min = df_clean[col].min()
    col_max = df_clean[col].max()
    df_clean[f"{col}_norm"] = (df_clean[col] - col_min) / (col_max - col_min)

# Drop any NaN in normalized columns
df_clean = df_clean.dropna(subset=["extraversion_norm", "agreeableness_norm", "minimalism_norm"])

print(f"Pool of {len(df_clean):,} participants with complete trait profiles")
print()

# Define 5 archetype profiles for influencer wealth research
target_profiles = {
    "P1_Status_Aspirational": {
        "extra": 0.85,  # Very extraverted (socially engaged)
        "agree": 0.20,  # Low agreeableness (not easily influenced)
        "min": 0.15,    # Anti-minimalist (materialistic)
        "description": "Engaged with influencer culture, aspires to luxury lifestyle"
    },
    "P2_Anti_Materialist": {
        "extra": 0.15,  # Introverted (less social media engagement)
        "agree": 0.80,  # Highly agreeable (values community over status)
        "min": 0.85,    # Strongly minimalist
        "description": "Rejects conspicuous consumption, values simplicity"
    },
    "P3_Social_Moderate": {
        "extra": 0.50,  # Moderately extraverted
        "agree": 0.50,  # Moderately agreeable
        "min": 0.50,    # Moderate on materialism
        "description": "Balanced perspective, context-dependent reactions"
    },
    "P4_Critical_Observer": {
        "extra": 0.75,  # Extraverted (notices influencer content)
        "agree": 0.25,  # Low agreeableness (skeptical, critical)
        "min": 0.30,    # Somewhat materialistic but not extreme
        "description": "Aware of influencer culture but critical of wealth displays"
    },
    "P5_Conflicted_Minimalist": {
        "extra": 0.85,  # Very extraverted (highly engaged socially)
        "agree": 0.30,  # Low agreeableness (independent thinker)
        "min": 0.85,    # Strongly minimalist
        "description": "THEORETICALLY INTERESTING: Socially engaged but anti-materialist"
    }
}

selected_ids = []
selected_participants = []

for profile_name, targets in target_profiles.items():
    available = df_clean[~df_clean["pid"].isin(selected_ids)].copy()

    if len(available) == 0:
        print(f"⚠️  No participants available for {profile_name}")
        continue

    # Calculate Euclidean distance to target profile
    available["distance"] = np.sqrt(
        (available["extraversion_norm"].astype(float) - targets["extra"]) ** 2 +
        (available["agreeableness_norm"].astype(float) - targets["agree"]) ** 2 +
        (available["minimalism_norm"].astype(float) - targets["min"]) ** 2
    ).astype(float)

    # Safety check
    if available["distance"].dtype == 'object':
        available["distance"] = pd.to_numeric(available["distance"], errors='coerce')
        available = available.dropna(subset=["distance"])

    if len(available) == 0:
        print(f"⚠️  All participants eliminated for {profile_name}")
        continue

    # Select best match
    best = available.nsmallest(1, "distance").iloc[0]
    selected_ids.append(best["pid"])
    selected_participants.append({
        "profile": profile_name,
        "description": targets["description"],
        "pid": best["pid"],
        "extraversion": best["extraversion"],
        "extra_pct": best["extra_pct"],
        "agreeableness": best["agreeableness"],
        "agree_pct": best["agree_pct"],
        "minimalism": best["minimalism"],
        "min_pct": best["min_pct"],
        "distance": best["distance"],
        "persona_summary": best["persona_summary"],
    })
    print(f"✓ {profile_name}: PID {best['pid']} (match quality: {best['distance']:.3f})")

print()
print("=" * 70)
print(f"SUCCESSFULLY SELECTED {len(selected_participants)} PARTICIPANTS")
print("=" * 70)
print()

# Display selected participants
for p in selected_participants:
    print(f"PROFILE: {p['profile']}")
    print(f"  {p['description']}")
    print(f"  PID: {p['pid']}")
    print(f"  Extraversion:  {p['extraversion']:.2f} ({p['extra_pct']}th percentile)")
    print(f"  Agreeableness: {p['agreeableness']:.2f} ({p['agree_pct']}th percentile)")
    print(f"  Minimalism:    {p['minimalism']:.2f} ({p['min_pct']}th percentile)")
    print(f"  Match quality: {p['distance']:.3f} (0 = perfect match)")
    print("-" * 70)
    print()

print()
print("=" * 70)
print("THEORETICAL MAPPING TO INFLUENCER WEALTH PERCEPTION")
print("=" * 70)
print()
print("P1 (Status Aspirational): High extraversion + Low minimalism")
print("  → Likely to ADMIRE influencer wealth displays")
print("  → Views luxury as aspirational and desirable")
print()
print("P2 (Anti-Materialist): Low extraversion + High minimalism")
print("  → Likely to REJECT influencer wealth displays")
print("  → Values simplicity and authenticity over status")
print()
print("P3 (Social Moderate): Balanced on all traits")
print("  → Context-dependent reactions")
print("  → May appreciate tasteful displays, reject excess")
print()
print("P4 (Critical Observer): High extraversion + Low agreeableness")
print("  → AWARE of influencer content but SKEPTICAL")
print("  → May see wealth displays as inauthentic or manipulative")
print()
print("P5 (Conflicted Minimalist): High extraversion + High minimalism")
print("  → THE THEORETICALLY INTERESTING CASE")
print("  → Socially engaged BUT anti-materialist")
print("  → May feel tension between social norms and personal values")
print("  → Could show ambivalent or complex reactions")
print()
print("=" * 70)
print("These profiles replace the Phase 1 fictional personas:")
print("  P1 ≈ Zoe (aspirational)")
print("  P2 ≈ Marcus (critical/anti-materialist)")
print("  P3 ≈ Priya (moderate/balanced)")
print("  P4 ≈ Derek (analytical/skeptical)")
print("  P5 ≈ Aaliyah (conflicted critic)")
print("=" * 70)
print()
print("✓ Ready for Phase 2 RAG interviews!")

In [ ]:
# CELL 2: Enter your OpenAI API key
# When you run this cell, a box will appear. Paste your key there and press Enter.
# Your key will NOT be visible on screen.

import getpass
import openai

api_key = getpass.getpass("")
client = openai.OpenAI(api_key=api_key)
print("")

In [ ]:
# CELL 6: Run RAG-grounded influencer wealth perception interviews
#
# THE RAG MECHANISM:
# Each participant's persona_summary (~2,000 words) becomes the GPT-4 system prompt.
# GPT-4 reads their full psychological profile BEFORE answering any question.
# The questions are IDENTICAL to Phase 1 for direct comparison.
#
# Runtime: ~1-2 minutes.

interview_questions = [
    "Q1: How do you react when an influencer you follow on TikTok or Instagram posts content "
    "showing expensive purchases, luxury travel, or a lavish lifestyle?",

    "Q2: Does it matter to you WHERE an influencer's wealth comes from? "
    "For example: does it change how you feel if the money comes from brand deals, "
    "family wealth, debt, or a separate job outside of social media?",

    "Q3: Have you ever unfollowed, criticized, or changed your purchasing behavior "
    "toward an influencer specifically because of how they displayed their wealth? "
    "Tell me what happened."
]

system_prompt_prefix = """You are a research participant in an interview study about social media influencers and wealth perception.
Your complete psychological profile, based on validated survey instruments, is provided below.
Answer the interviewer's questions authentically and in character, consistent with your profile.
Speak in first person, naturally, as this real person would -- not as a description of them.

Your responses should reflect your actual scores on extraversion, agreeableness, minimalism,
conscientiousness, openness, neuroticism, financial literacy, and all other measured characteristics.

Be specific and personal. Do not be generic. Draw on your personality traits when forming opinions.

For reference on how your traits might influence your views:
- Your EXTRAVERSION score reflects how socially engaged and outgoing you are
- Your AGREEABLENESS score reflects how trusting vs. skeptical you are of others
- Your MINIMALISM score reflects your attitude toward materialism and consumption

YOUR COMPLETE PSYCHOLOGICAL PROFILE:
========================
"""

print("Running Phase 2 RAG-grounded influencer wealth perception interviews...")
print("(GPT-4 is reading each participant's full psychological profile before answering)")
print()

results = []

for i, participant in enumerate(selected_participants, 1):
    print(f"Interviewing participant {i}/5: PID {participant['pid']} ({participant['profile']})...")

    user_message = "\n\n".join([
        f"{q}\n(Please answer this question fully before moving to the next.)"
        for q in interview_questions
    ])

    system_message = system_prompt_prefix + participant["persona_summary"]

    response = client.chat.completions.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user",   "content": user_message}
        ],
        temperature=0.7,
        max_tokens=1200
    )

    answer = response.choices[0].message.content
    results.append({
        "profile":      participant["profile"],
        "description":  participant["description"],
        "pid":          participant["pid"],
        "extra_pct":    participant["extra_pct"],
        "agree_pct":    participant["agree_pct"],
        "min_pct":      participant["min_pct"],
        "response":     answer
    })
    print(f"  Done.")

print()
print("All 5 interviews complete. Run the next cell to see results.")

In [ ]:
# CELL 7: Display results with Phase 1 comparison guidance

print("=" * 70)
print("PHASE 2 RAG-GROUNDED INFLUENCER WEALTH PERCEPTION INTERVIEW RESULTS")
print("Twin-2K-500 | Toubia et al. (2025)")
print("=" * 70)

profile_expectations = {
    "P1_Status_Aspirational": "HIGH Extraversion (87th%ile) | LOW Agreeableness (1st%ile) | LOW Minimalism (5th%ile) → Expected: Admires wealth displays",
    "P2_Anti_Materialist": "LOW Extraversion (12th%ile) | HIGH Agreeableness (68th%ile) | HIGH Minimalism (86th%ile) → Expected: Rejects conspicuous consumption",
    "P3_Social_Moderate": "MODERATE Extraversion (57th%ile) | MODERATE Agreeableness (11th%ile) | MODERATE Minimalism (30th%ile) → Expected: Context-dependent",
    "P4_Critical_Observer": "HIGH Extraversion (87th%ile) | LOW Agreeableness (2nd%ile) | LOW Minimalism (5th%ile) → Expected: Aware but skeptical",
    "P5_Conflicted_Minimalist": "HIGH Extraversion (95th%ile) | LOW Agreeableness (7th%ile) | HIGH Minimalism (95th%ile) → Expected: Conflicted/ambivalent",
}

for r in results:
    print()
    print(f"PARTICIPANT: PID {r['pid']}")
    print(f"Profile:     {r['profile']}")
    print(f"             {r['description']}")
    print(f"Theory:      {profile_expectations.get(r['profile'], 'N/A')}")
    print(f"Trait Scores: Extraversion={r['extra_pct']}th | Agreeableness={r['agree_pct']}th | Minimalism={r['min_pct']}th")
    print()
    print(r["response"])
    print()
    print("-" * 70)

print()
print("=" * 70)
print("PHASE 1 vs PHASE 2 COMPARISON GUIDE")
print("=" * 70)
print()
print("In Phase 1, you wrote fictional personas and their responses.")
print("In Phase 2, GPT-4 responded AS real participants based on their full profiles.")
print()
print("COMPARE:")
print("1. Are the response patterns consistent with trait predictions?")
print("   - Does high extraversion lead to more social media engagement?")
print("   - Does low agreeableness lead to more skepticism?")
print("   - Does high minimalism lead to rejection of wealth displays?")
print()
print("2. How does P5 (Conflicted Minimalist) respond?")
print("   - This profile (high extraversion + high minimalism) creates tension")
print("   - Do they show ambivalence or complex reasoning?")
print("   - This is your THEORETICALLY INTERESTING case")
print()
print("3. Are the responses more nuanced than Phase 1?")
print("   - Do they reference specific traits from their profile?")
print("   - Do they show realistic complexity vs. one-dimensional responses?")
print()
print("=" * 70)
print("NEXT STEPS FOR YOUR RESEARCH")
print("=" * 70)
print()
print("For your seminar paper or presentation, you can:")
print()
print("1. THEMATIC ANALYSIS:")
print("   - Code responses for: admiration, rejection, skepticism, ambivalence")
print("   - See if coding matches trait predictions")
print()
print("2. QUOTE EXTRACTION:")
print("   - Pull representative quotes from each profile type")
print("   - Use in your analysis section")
print()
print("3. THEORETICAL CONTRIBUTION:")
print("   - How does extraversion + minimalism interaction predict responses?")
print("   - Is this captured in existing influencer marketing literature?")
print("   - Does the P5 profile suggest a new consumer segment?")
print()
print("4. METHODOLOGICAL REFLECTION:")
print("   - What did Phase 2 reveal that Phase 1 could not?")
print("   - How does RAG improve on fictional personas?")
print("   - What are the limitations of GPT-4 as a digital twin?")
print()
print("=" * 70)